# Advent of Code 2025 - Day 7

https://adventofcode.com/2025/day/7

## Imports

In [ ]:
import marimo as mo

from collections import defaultdict

from aoc.utils import read_data

## Day 7 - Laboratories

### Part 1 - Solution

In [ ]:
def exercise_7_1_find_number_of_beam_splits(tachyon_manifold: list[str]) -> int:
    """Count the total number of times tachyon beams are split in a manifold.

    A tachyon beam starts at the location marked 'S' in the top row and moves
    downward. Beams pass freely through empty spaces ('.') and are split when
    they encounter a splitter ('^'). When a splitter is hit, the beam stops and
    two new beams are emitted from the immediate left and right positions of the
    splitter. Overlapping beams in the same column are counted only once per
    row.

    Args:
        tachyon_manifold (list[str]): A list of strings representing the grid of
            the tachyon manifold. 'S' marks the starting point, '.' is empty
            space, and '^' is a splitter.

    Returns:
        int: The total number of beam splits that occur in the manifold.
    """
    beam_split_counter: int = 0
    beam_x_coordinates: set[int] = {tachyon_manifold[0].index("S")}

    for row_index in range(len(tachyon_manifold) - 1):
        new_beam_x_coordinates = set()

        for x in beam_x_coordinates:
            if tachyon_manifold[row_index + 1][x] == "^":
                new_beam_x_coordinates.add(x - 1)
                new_beam_x_coordinates.add(x + 1)
                beam_split_counter += 1
            else:
                new_beam_x_coordinates.add(x)

        beam_x_coordinates = new_beam_x_coordinates

    return beam_split_counter

In [ ]:
example_tachyon_manifold = [
    ".......S.......",
    "...............",
    ".......^.......",
    "...............",
    "......^.^......",
    "...............",
    ".....^.^.^.....",
    "...............",
    "....^.^...^....",
    "...............",
    "...^.^...^.^...",
    "...............",
    "..^...^.....^..",
    "...............",
    ".^.^.^.^.^...^.",
    "...............",
]

solution_example_7_1 = exercise_7_1_find_number_of_beam_splits(
    tachyon_manifold=example_tachyon_manifold
)

assert solution_example_7_1 == 21

In [ ]:
tachyon_manifold = read_data(file_path="data/2025/2025_day_07.txt", separator="\n")

solution_7_1 = exercise_7_1_find_number_of_beam_splits(
    tachyon_manifold=tachyon_manifold
)

assert solution_7_1 == 1590

### Part 2 - Solution

In [ ]:
def exercise_7_2_find_number_of_timelines(tachyon_manifold: list[str]) -> int:
    """Count the total number of quantum timelines for a single tachyon
    particle traversing a quantum tachyon manifold using the many-worlds
    interpretation.

    Each splitter ('^') splits the current timeline into two: one going left
    and one going right. Timelines do not merge; every possible path is
    counted as a distinct timeline. The particle starts at the location
    marked 'S' in the top row and moves downward, creating new timelines at
    each splitter.

    Args:
        tachyon_manifold (list[str]): A list of strings representing the
            quantum tachyon manifold. 'S' marks the starting point, '.' is
            empty space, and '^' is a splitter.

    Returns:
        int: Total number of unique timelines after the particle traverses
            the manifold.
    """
    timeline_counter = {tachyon_manifold[0].index("S"): 1}

    for i in range(len(tachyon_manifold) - 1):
        beam_counter_per_row = defaultdict(int)

        for x, count in timeline_counter.items():
            if tachyon_manifold[i + 1][x] == "^":
                beam_counter_per_row[x - 1] += count
                beam_counter_per_row[x + 1] += count
            else:
                beam_counter_per_row[x] += count

        timeline_counter = beam_counter_per_row

    total_number_of_timelines = sum(timeline_counter.values())
    return total_number_of_timelines

In [ ]:
solution_example_7_2 = exercise_7_2_find_number_of_timelines(
    tachyon_manifold=example_tachyon_manifold
)

assert solution_example_7_2 == 40

In [ ]:
solution_7_2 = exercise_7_2_find_number_of_timelines(
    tachyon_manifold=tachyon_manifold
)

assert solution_7_2 == 20571740188555

In [ ]:
def exercise_7_2_find_number_of_timelines_iterative(tachyon_manifold: list[str]) -> int:
    """Count the total number of quantum timelines for a single tachyon particle
    traversing a quantum tachyon manifold using an iterative DFS approach.

    A single tachyon particle starts at 'S' in the top row and moves downward.
    Each splitter ('^') creates two new timelines: one moving left and one
    moving right. Empty cells ('.') allow the particle to continue straight
    down. This function explores all possible paths iteratively, using
    backtracking to account for all splits.

    Args:
        tachyon_manifold (list[str]): A list of strings representing the quantum
            tachyon manifold. 'S' marks the starting point, '.' is empty space,
            and '^' is a splitter.

    Returns:
        int: Total number of unique timelines after the particle traverses the
            manifold.
    """

    def propagate_beam_downward(
        beam_x_coordinates: list[int],
        choices: list[str],
        tachyon_manifold: list[str],
    ) -> tuple[list[int], list[str]]:
        """Propagate the beam downward through the manifold until the bottom is
        reached.

        For each row below the current beam position:
        - If the cell is a splitter ('^'), a left branch is chosen and recorded
          in choices.
        - If the cell is empty ('.'), the beam continues straight down.

        Args:
            beam_x_coordinates (list[int]): Current x-coordinates of the beam
                path.
            choices (list[str]): List of choices taken so far ('L', 'R', '-').
            tachyon_manifold (list[str]): The full quantum tachyon manifold
                grid.

        Returns:
            tuple[list[int], list[str]]: Updated beam x-coordinates and choices
                after propagating to the bottom of the manifold.
        """
        while len(beam_x_coordinates) < len(tachyon_manifold):
            x = beam_x_coordinates[-1]
            char_below = tachyon_manifold[len(beam_x_coordinates)][x]

            # Always go left when encountering a splitter.
            if char_below == "^":
                choices.append("L")
                beam_x_coordinates.append(x - 1)
            elif char_below == ".":
                choices.append("-")
                beam_x_coordinates.append(x)

        return beam_x_coordinates, choices

    timeline_counter = 1
    choices = []
    beam_x_coordinates = [tachyon_manifold[0].index("S")]

    beam_x_coordinates, choices = propagate_beam_downward(
        beam_x_coordinates, choices, tachyon_manifold
    )

    while True:
        for i in range(len(beam_x_coordinates) - 1, -1, -1):
            if choices[i - 1] == "L":
                # Backtrack to the previous choice.
                beam_x_coordinates = beam_x_coordinates[:i].copy()
                choices = choices[: i - 1].copy()

                # Now take the right branch instead.
                choices.append("R")
                beam_x_coordinates.append(beam_x_coordinates[-1] + 1)

                beam_x_coordinates, choices = propagate_beam_downward(
                    beam_x_coordinates, choices, tachyon_manifold
                )

                timeline_counter += 1
                break
        else:
            break

    return timeline_counter

In [ ]:
solution_example_7_2_v2 = exercise_7_2_find_number_of_timelines_iterative(
    tachyon_manifold=example_tachyon_manifold
)

assert solution_example_7_2_v2 == 40

In [ ]:
# I think this code will give the correct answer, however because of the iterative approach,
# and the scaling with 2 to the power of the number of rows in the Tachyon manifold diagram,
# this will take way too much time. I don't know how long exactly, but it's not a feasible solution.

# exercise_7_2_find_number_of_timelines_iterative(tachyon_manifold=tachyon_manifold)